In [1]:
!pip install koreanize_matplotlib

import warnings
import koreanize_matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ast

# 경고 무시
warnings.filterwarnings("ignore")
%config lnlineBackend.figure_format = 'retina'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # 출력할 너비를 넉넉하게 조정
pd.set_option('display.expand_frame_repr', False)  # 옆으로 길어져도 줄바꿈 없이 출력
pd.set_option('display.max_colwidth', None)  # 긴 문자열도 생략 없이 출력

try:
	from google.colab import drive
	drive.mount('/content/drive')

	import os
	os.chdir('/content/drive/MyDrive/파트4')
	print('✅ Succesful access google_drive_directory')
	
except Exception as e:
	print('🤗 Hello vscode')

🤗 Hello vscode


In [14]:
API_KEY_PATH ='/home/project_yujin/API_KEY/sprintda03-yujin.json'

## get_df 함수
def get_df(db_name, table_name):
    if table_name in ['accounts_user', 'accounts_blockrecord']:
        table_name = pd.read_parquet(
            f"gs://high_project/{db_name}/{table_name}.parquet", 
            storage_options={'token' : API_KEY_PATH})
    else:
        table_name = pd.read_csv(
            f"gs://high_project/{db_name}/{table_name}.csv",
            storage_options={'token' : API_KEY_PATH}
            )
    return table_name
    

## literal_eval 형변환 함수
import ast
def to_literal_eval(df, column):
    return  df[column].apply(lambda x: ast.literal_eval(x) if x != '[]' else [])


drop_users = [831956, 1580627, 1580689, 1580626, 995177]
drop_schools = [5948, 5949, 5964]

# 리스트 내에 드랍 유저가 있는지 확인하는 함수
def find_drop_users(df, column):
    drop_users = [831956, 1580627, 1580689, 1580626, 995177]
    for i in drop_users:
        count_drop_rows = len(df[df[column].apply(lambda x: i in x)])
        if count_drop_rows != 0:
            print(f"‼️ 관리자 {i}가 포함된 행 {count_drop_rows}개 존재")
        else:
            print(f"✅ 관리자 {i} 포함행 없음")

## accounts_user

In [6]:
accounts_user = get_df('votes','accounts_user')

In [9]:
accounts_user.head(1)

,id,is_superuser,is_staff,gender,point,friend_id_list,is_push_on,created_at,block_user_id_list,hide_user_id_list,ban_status,report_count,alarm_count,pending_chat,pending_votes,group_id
1,831962,0,0,F,2248,"[833025, 832642, 982531, 879496, 838541, 837521, 833041, 832151, 1082907, 1426466, 1541413, 1577131, 837806, 834486, 834358, 1575225, 1576252, 837950, 1446852, 1577930, 841037, 1577938, 832340, 831958, 849624, 837338, 1577954, 849763, 862823, 1577703, 834415, 833009, 834289, 833011, 842865, 833013, 856050, 833017, 833018, 833022, 833023, 1580476, 1580855]",1,2023-03-29 05:18:56.162368,[],[],N,253,40878,5499,110,12.0


In [38]:
# 관리자 제거
drop_users = [831956, 1580627, 1580689, 1580626, 995177]
accounts_user = accounts_user[~accounts_user['id'].isin(drop_users)]

# 관리자 관련 컬럼 제거
accounts_user = accounts_user.drop(columns=['is_superuser', 'is_staff'])

In [10]:
# 유저별 친구수, 차단친구수, 숨김친구수 컬럼 추가하기

# 1. 리스트형으로 변환
accounts_user['friend_id_list'] = to_literal_eval(accounts_user, 'friend_id_list')
accounts_user['block_user_id_list'] = to_literal_eval(accounts_user, 'block_user_id_list')
accounts_user['hide_user_id_list'] = to_literal_eval(accounts_user, 'hide_user_id_list')

In [15]:
# 2. 리스트내 관리자가 있는지 확인
find_drop_users(accounts_user, 'friend_id_list')

‼️ 관리자 831956가 포함된 행 6개 존재
✅ 관리자 1580627 포함행 없음
✅ 관리자 1580689 포함행 없음
✅ 관리자 1580626 포함행 없음
✅ 관리자 995177 포함행 없음


In [16]:
find_drop_users(accounts_user, 'block_user_id_list')

✅ 관리자 831956 포함행 없음
✅ 관리자 1580627 포함행 없음
✅ 관리자 1580689 포함행 없음
✅ 관리자 1580626 포함행 없음
✅ 관리자 995177 포함행 없음


In [17]:
find_drop_users(accounts_user, 'hide_user_id_list')

✅ 관리자 831956 포함행 없음
✅ 관리자 1580627 포함행 없음
✅ 관리자 1580689 포함행 없음
✅ 관리자 1580626 포함행 없음
✅ 관리자 995177 포함행 없음


In [ ]:
# 2-1. friend_id_list에 831956 관리자가 포함되었던 행 확인
accounts_user[accounts_user['friend_id_list'].apply(lambda x: 831956 in x)]

# 관리자를 친구로 둔 유저 6명 발견 ['913158', '1043565', '1064695', '1274736', '1292473', '1488461']

In [31]:
# 2-2. 관리자가 친구인 행을 삭제 vs 친구리스트에서 관리자 아이디만 삭제

# - 관리자가 친구인 행을 삭제
drop_index = accounts_user[accounts_user['friend_id_list'].apply(lambda x: 831956 in x)].index.tolist()
accounts_user.drop(index=drop_index)

# - 친구리스트에서 관리자 아이디만 삭제
def delete_drop_user(df, column, drop_user):
    return df[[column]].apply(lambda x: [i for i in x if i != drop_user])

delete_drop_user(accounts_user, 'friend_id_list', 831956)

# 임의 채택 친구리스트에서 관리자 아이디만 삭제
accounts_user['friend_id_list'] = delete_drop_user(accounts_user, 'friend_id_list', 831956)

In [32]:
# 유저별 친구수, 차단유저수, 숨김유저수 
accounts_user['count_friends'] = accounts_user['friend_id_list'].apply(len)
accounts_user['count_block_users'] = accounts_user['block_user_id_list'].apply(len)
accounts_user['count_hide_users'] = accounts_user['hide_user_id_list'].apply(len)

In [40]:
accounts_user.drop(columns=['friend_id_list', 'block_user_id_list', 'hide_user_id_list']).head(5)

,id,gender,point,is_push_on,created_at,ban_status,report_count,alarm_count,pending_chat,pending_votes,group_id,count_friends,count_block_users,count_hide_users
1,831962,F,2248,1,2023-03-29 05:18:56.162368,N,253,40878,5499,110,12.0,43,0,0
2,832151,M,1519,0,2023-03-29 12:56:34.989468,N,0,37,0,47,1.0,51,0,0
3,832340,F,57,1,2023-03-29 12:56:35.020790,N,0,19,0,21,1.0,57,0,0
4,832520,M,1039,0,2023-03-29 12:56:35.049311,N,0,29,0,15,12.0,18,0,0
5,832614,M,1048,1,2023-03-29 12:56:35.064406,N,0,28,0,14,12.0,21,0,0


In [ ]:
accounts_user

**유저 코호트 분석**
- 코호트를 나누는 기준은?
    - 친구수, 알람수에 따른 인기유저코호트 vs 비인기유저코호트
    - 가입 시기에 따른 코호트
    - 학교별 코호트

- 코호트를 나누어서 볼것은?
    - 유지율
    - 이탈률

- 코호트 분석을 통해 알고 싶은것
    - 친구간의 네트워크가 유지율, 이탈율에 영향을 줄지 
    - 가입 시기가 유지율 이탈율에 영향을 줄지 => 유지율이 높았던 가입시기의 코호트에 어떤 마케팅을 했었는지 => 알수는 없음..
    - 학교의 위치(서울시 vs 제주시)가 유지율과 이탈율에 영향을 줄지 즉 유저가 사는 환경또는 문화가 서비스에 영향을 줄지 => 마케팅

# 2) accounts_blockrecord

In [42]:
accounts_blockrecord = get_df('votes','accounts_blockrecord')

## 23_hackle_properties

In [304]:
hackle_properties = get_df('hackle','hackle_properties')

In [305]:
# 유저 user_id 결측치 제거
hackle_properties.dropna(subset=['user_id'], inplace=True)

# 유저아이디가 문자인것은 제외
hackle_properties = hackle_properties[hackle_properties['user_id'].apply(lambda x: x.isdigit())]

# 관리자 제거
hackle_properties = hackle_properties[~hackle_properties['user_id'].isin(drop_users)]

# 순서 유저별로 보기 좋게
hackle_properties = hackle_properties[['id', 'user_id' ,'session_id' ,'device_id', 'language', 'osname', 'osversion' ,'versionname']]

In [277]:
hackle_properties['language_main'] = hackle_properties['language'].apply(lambda x: x.split('-')[0])
hackle_properties['language_region'] = hackle_properties['language'].apply(lambda x: x.split('-')[1] if '-' in x else None)

In [308]:
hackle_properties['user_id'].nunique()

230853

In [307]:
hackle_properties.groupby(['user_id'])['osname'].nunique().reset_index()[hackle_properties.groupby(['user_id'])['osname'].nunique().reset_index()['osname'] > 1]

,user_id,osname
863,1003125,2
2100,1006500,2
2117,1006548,2
2538,1007746,2
2554,1007777,2
...,...,...
229796,995845,2
229887,996155,2
229908,996234,2
230819,999899,2


In [278]:
len(hackle_properties)

334091

In [279]:
hackle_properties = hackle_properties.sort_values(by=['user_id','osversion'])
hackle_properties = hackle_properties[~hackle_properties.loc[:, ['user_id', 'session_id' ,'device_id', 'language', 'osname', 'osversion', 'language_main', 'language_region']]\
    .duplicated(keep='last')]

In [280]:
hackle_properties = hackle_properties.sort_values(by=['user_id','versionname'])
hackle_properties = hackle_properties[~hackle_properties.loc[:, ['user_id', 'session_id' ,'device_id', 'language', 'osname', 'versionname', 'language_main', 'language_region']]\
    .duplicated(keep='last')]

In [281]:
len(hackle_properties)

237921

In [294]:
hackle_properties = hackle_properties.sort_values(by=['user_id', 'device_id', 'versionname'])
hackle_properties = hackle_properties[~hackle_properties.loc[:, ['user_id', 'language', 'osname', 'language_main', 'language_region']]\
    .duplicated(keep='last')]

In [295]:
len(hackle_properties)

232462

In [303]:
hackle_properties[hackle_properties.loc[:, ['user_id', 'language_main']].duplicated(keep=False)]['user_id'].value_counts()

,count
user_id,
1003125,2
1472749,2
1491511,2
1486219,2
1485763,2
...,...
1199184,2
1196868,2
1195692,2


In [232]:
df = hackle_properties.loc[:, ['user_id', 'session_id', 'language', 'osname', 'osversion', 'device_id']].drop_duplicates()

In [233]:
df[df['user_id'].duplicated(keep=False)].sort_values(by='user_id')

,user_id,session_id,language,osname,osversion,device_id
339071,00M3Y1SoydeYioCEhbtRqrJLHA72,00M3Y1SoydeYioCEhbtRqrJLHA72,ko,Android,11,086d0e8e-d577-4576-ac54-62cc02f033e9
369217,00M3Y1SoydeYioCEhbtRqrJLHA72,00M3Y1SoydeYioCEhbtRqrJLHA72,ko-KR,iOS,16.4.1,03E0DDA4-D5DA-4F66-AD1F-6B368A779AAC
299653,05MIATQZQbg87FYBnN0kOIqWCN13,05MIATQZQbg87FYBnN0kOIqWCN13,ko-KR,iOS,16.6,D2DB03E5-E433-48F4-8500-F621C846E0D0
144682,05MIATQZQbg87FYBnN0kOIqWCN13,05MIATQZQbg87FYBnN0kOIqWCN13,ko-KR,iOS,16.5.1,D2DB03E5-E433-48F4-8500-F621C846E0D0
439313,09DSx11HBwfVaBPMhshdTpGvdIv2,09DSx11HBwfVaBPMhshdTpGvdIv2,ko-KR,iOS,16.6,2EDE0375-B3F8-4E2A-AC95-673C4BF3AFF2
...,...,...,...,...,...,...
351068,zq2SDFFQwoPVUkYiF4kDSC0RVVM2,65CEE76E-C9EB-483C-BD48-462BEF95ACDB,ko-KR,iOS,16.5.1,65CEE76E-C9EB-483C-BD48-462BEF95ACDB
414827,ztCGF6QjLgYtCrf6XhI3HAytZO93,ztCGF6QjLgYtCrf6XhI3HAytZO93,ko-KR,iOS,16.5.1,8646DEEB-58C8-48AC-B564-FC36E7EE9D25
375830,ztCGF6QjLgYtCrf6XhI3HAytZO93,ztCGF6QjLgYtCrf6XhI3HAytZO93,ko-KR,iOS,16.6,8646DEEB-58C8-48AC-B564-FC36E7EE9D25
439259,ztNtAcyTlfV6X1WNp2WTJnXAysv2,ztNtAcyTlfV6X1WNp2WTJnXAysv2,en-KR,iOS,16.5.1,179A2A23-D221-4EB6-A891-899B953E7CEE


In [224]:
hackle_properties

,id,session_id,user_id,language,osname,osversion,versionname,device_id
1,2,8QXy31PQxbW9qLzq0Y1dhR8Ypm52,1046711,ko-KR,iOS,16.5.1,2.0.3,D5417226-F71B-4A9E-A180-CD072F2AB279
2,3,6bcea65d-9f40-46fc-888c-700fe707483f,1545130,ko,Android,13,2.0.5,6bcea65d-9f40-46fc-888c-700fe707483f
3,4,XVYNT6zfhFWqIg9omwg2AHDjTLx2,1224793,ko,Android,13,2.0.5,a05c1595-3e05-434b-8684-218b528bd725
4,5,XFB2SPiGfjbVhvJ3Q3DBsaT3m2B3,1329450,ko-US,iOS,16.5.1,2.0.5,EAC6C0B3-7CE8-40EA-8A91-9977C0BA5EF3
5,6,LztzUUFoRxdqTSPgQrX3MAAyNkM2,LztzUUFoRxdqTSPgQrX3MAAyNkM2,ko-KR,iOS,16.1,2.0.5,3F199073-9390-4137-B0B0-0DC4FC103009
...,...,...,...,...,...,...,...,...
525344,525345,b82eptestoYIkel7zGItYz9XqF43,902597,ko-KR,iOS,16.0,2.0.3,B59DCE74-59FB-4417-9DDB-F9B620D71DFC
525345,525346,KlGJxOfY4XdbxnwzPckMh4NdwBk2,1373831,ko-KR,iOS,16.5.1,2.0.5,2EB9127D-703A-495B-8A1E-6667ACA9E724
525346,525347,HGxbSi2oq4MdFVGdQx2UH3f9Aq73,1043127,en-KR,iOS,16.1.2,2.0.3,4400D84D-0353-49C9-818E-6A45D54F1039
525348,525349,gQ5GvGk7kGWwnQbOzQ8fxseQp8B2,7m8IwV5H3aaxr1bdPbbkvvJMvtf2,ko,Android,12,1.2.15,eca30324-3b48-41a6-9628-ff663896dd23


In [221]:
hackle_properties['user_id'].nunique()

327380

In [209]:
hackle_properties.loc[:, 'user_id': 'versionname'].duplicated().sum()

0

In [ ]:
hackle_properties

### 유저별 방문 카운트

In [ ]:
## 재방문했다는것을 유니크한 session_id값의 개수를 봐야하나
# 아니면 value_counts()

In [207]:
hackle_properties[hackle_properties['user_id'].duplicated(keep=False)].sort_values(by='user_id', ascending=True)

,id,user_id,session_id,device_id,language,osname,osversion,versionname,language_main,language_region
387299,387300,1000009,tTCMTeTwQ9S7pvRPYSX7ZO7d6hb2,594ec0e3-1eb2-4e42-b864-434f6cc64014,ko,Android,12,2.0.3,ko,None
346164,346165,1000009,tTCMTeTwQ9S7pvRPYSX7ZO7d6hb2,594ec0e3-1eb2-4e42-b864-434f6cc64014,ko,Android,12,2.0.5,ko,None
80006,80007,1000013,iaXtp90djrhuhk3vWk7Bp6mqyvI3,D443A78F-C7AE-46E1-A8E5-2461194C7859,ko-KR,iOS,16.5.1,2.0.3,ko,KR
226599,226600,1000013,iaXtp90djrhuhk3vWk7Bp6mqyvI3,D443A78F-C7AE-46E1-A8E5-2461194C7859,ko-KR,iOS,16.5.1,2.0.5,ko,KR
295254,295255,1000028,3dNw55G8Gnf7QzZQk8cKdhQzkTJ2,a298fe2e-1475-4f61-8496-94d0e925e7d3,ko,Android,13,2.0.5,ko,None
...,...,...,...,...,...,...,...,...,...,...
355039,355040,999976,jCFyPxnoyQYOtSGHy8uFrroKRW03,bc9ff767-f8a5-4ea1-a5ae-b258568aaed2,ko,Android,13,2.0.5,ko,None
167494,167495,999990,pW222QK4eocZ8WaTsJACHIRL9ov1,A82C1509-2521-4D5E-BC8C-580E23D825B6,ko-KR,iOS,16.5.1,2.0.5,ko,KR
480451,480452,999990,pW222QK4eocZ8WaTsJACHIRL9ov1,A82C1509-2521-4D5E-BC8C-580E23D825B6,ko-KR,iOS,16.5.1,2.0.3,ko,KR
9799,9800,999992,9b590cfa-cffa-4c35-8132-82b160032444,9b590cfa-cffa-4c35-8132-82b160032444,ko,Android,13,2.0.3,ko,None


In [213]:
df = hackle_properties[hackle_properties['user_id'].duplicated(keep=False)].sort_values(by='user_id', ascending=True)
max_version_idx = df.groupby('user_id')['versionname'].idxmax()
df_latest = df.loc[max_version_idx].reset_index(drop=True)
df_latest

,id,user_id,session_id,device_id,language,osname,osversion,versionname,language_main,language_region
0,346165,1000009,tTCMTeTwQ9S7pvRPYSX7ZO7d6hb2,594ec0e3-1eb2-4e42-b864-434f6cc64014,ko,Android,12,2.0.5,ko,None
1,226600,1000013,iaXtp90djrhuhk3vWk7Bp6mqyvI3,D443A78F-C7AE-46E1-A8E5-2461194C7859,ko-KR,iOS,16.5.1,2.0.5,ko,KR
2,295255,1000028,3dNw55G8Gnf7QzZQk8cKdhQzkTJ2,a298fe2e-1475-4f61-8496-94d0e925e7d3,ko,Android,13,2.0.5,ko,None
3,6568,1000030,C8OI0sJUDfXhIXclVCZt0ekaE8E2,4c4fc4fd-9510-406c-b220-6983cbde5aa3,en,Android,13,2.0.5,en,None
4,362060,1000035,HeSXZkNz5SPTqNNxPV5BI29mFq22,cad23a43-11e9-4ee1-b263-3d79b677c7af,ko,Android,9,2.0.5,ko,None
...,...,...,...,...,...,...,...,...,...,...
83224,199928,999968,MGkDWvHkAIcMcK3PAbDtPPV5Fr62,55f13d29-2d81-4cb4-9cc8-0c84cb6a0276,ko,Android,12,2.0.5,ko,None
83225,95898,999971,bEYVNcufWRarmOX3TIuA8Zc1qGD2,A21E346E-F434-4A4F-BB6B-441DD1CD4152,ko-KR,iOS,15.4.1,2.0.5,ko,KR
83226,355040,999976,jCFyPxnoyQYOtSGHy8uFrroKRW03,bc9ff767-f8a5-4ea1-a5ae-b258568aaed2,ko,Android,13,2.0.5,ko,None
83227,167495,999990,pW222QK4eocZ8WaTsJACHIRL9ov1,A82C1509-2521-4D5E-BC8C-580E23D825B6,ko-KR,iOS,16.5.1,2.0.5,ko,KR


In [201]:
hackle_properties.groupby('user_id')['session_id'].nunique().reset_index()

,user_id,session_id
0,1000000,1
1,1000009,1
2,1000012,1
3,1000013,1
4,1000015,1
...,...,...
230848,999987,1
230849,999990,1
230850,999992,1
230851,999996,1


In [203]:
hackle_properties['user_id'].value_counts()

,count
user_id,
1578652,17
1459833,13
1571506,13
1388873,12
1285353,12
...,...
1176873,1
1189864,1
1512850,1


In [188]:
# print(hackle_properties['language_main'].unique())
language_count = hackle_properties['language_main'].value_counts().reset_index()
fig = px.pie(
    language_count,
    names='language_main',    # 라벨 표시할 컬럼
    values='count',      # 비율 계산할 수치 컬럼
    title='Language Distribution'
)
fig.show()

In [192]:
# print(hackle_properties['language_region'].unique())
region_count = hackle_properties['language_region'].value_counts().reset_index()
fig = px.pie(
    region_count,
    names='language_region',    # 라벨 표시할 컬럼
    values='count',      # 비율 계산할 수치 컬럼
    title='Language Distribution'
)
fig.show()

In [171]:
import matplotlib.pyplot as plt


In [172]:
df = hackle_properties['language'].value_counts().reset_index()
plt.bar(df, x='language', y='count')

TypeError: bar() got multiple values for argument 'x'

## 유저별 접속 세션 이력

In [162]:
hackle_properties['user_id'].value_counts().reset_index()

,user_id,count
0,1578652,17
1,1459833,13
2,1571506,13
3,1388873,12
4,1285353,12
...,...,...
230848,1176873,1
230849,1189864,1
230850,1512850,1
230851,1228261,1


In [52]:
len(hackle_properties['user_id'].unique())

327380

In [51]:
wow = [i for i in hackle_properties['user_id'].unique().tolist() if not i.isdigit()]
len(wow)

96527

In [55]:
(96527 / 327380)*100

29.484696682753984

### 유저아이디가 문자인건 제외

In [67]:
##
## 디바이스하나에 여러개의 세션 ㅋㅋ ㅠㅠ
hackle_properties.query("session_id == '021DAAC9-4B87-4086-ADD7-A85673F3E661'")

,id,session_id,user_id,language,osname,osversion,versionname,device_id
141152,141153,021DAAC9-4B87-4086-ADD7-A85673F3E661,1579184,ko-KR,iOS,16.5.1,2.0.3,021DAAC9-4B87-4086-ADD7-A85673F3E661
235683,235684,021DAAC9-4B87-4086-ADD7-A85673F3E661,1172318,ko-KR,iOS,16.5.1,2.0.3,021DAAC9-4B87-4086-ADD7-A85673F3E661
401896,401897,021DAAC9-4B87-4086-ADD7-A85673F3E661,1172318,ko-KR,iOS,16.5.1,2.0.0,021DAAC9-4B87-4086-ADD7-A85673F3E661


In [46]:
hackle_properties.query("session_id == '00057831-A672-4163-9C02-AB920A371F2C'")

,id,session_id,user_id,language,osname,osversion,versionname,device_id
282097,282098,00057831-A672-4163-9C02-AB920A371F2C,1548609,ko-KR,iOS,16.1.2,2.0.5,00057831-A672-4163-9C02-AB920A371F2C
518233,518234,00057831-A672-4163-9C02-AB920A371F2C,c5gLjsxgDkRXXlQlYQPH0CkosKf2,ko-KR,iOS,16.1.2,2.0.5,00057831-A672-4163-9C02-AB920A371F2C


In [47]:
hackle_properties.query("session_id == '0011244f-e78e-44b2-8736-d661426deba0'")

,id,session_id,user_id,language,osname,osversion,versionname,device_id
86725,86726,0011244f-e78e-44b2-8736-d661426deba0,1311153,ko,Android,13,2.0.5,0011244f-e78e-44b2-8736-d661426deba0
281714,281715,0011244f-e78e-44b2-8736-d661426deba0,TZlwJIxWdRRw78nfzFGwJdJIFT33,ko,Android,13,2.0.5,0011244f-e78e-44b2-8736-d661426deba0


In [ ]:
# 유저별 세션수
# 유저별 사용언어
# 유저별 운영체제
# 유저별 운영체제 버전
# 유저별 앱버전
# 유저별 사용 기기의 고유 id

## device_properties

In [34]:
device_properties = get_df('hackle','device_properties')
device_properties = device_properties[~device_properties['device_id'].duplicated(keep=False)]